# Reconstrução de Mapas de Campo Magnético $B_z$ — Downward Continuation + CNN

Pipeline para refinar e reconstruir mapas de campo magnético $B_z(x, y)$ medidos
por magnetometria de varredura, em diferentes alturas sensor–amostra.

**Etapas:**
1. Funções físicas no domínio de Fourier (grade $k$, downward continuation, deconvolução).
2. Carregar e visualizar um mapa.
3. Demonstrar a continuação para baixo em um mapa.
4. Montar o conjunto de dados (carregar todos os mapas + *data augmentation*).
5. Gerar pares entrada → alvo.
6. Recortar em *patches* 2D.
7. `Dataset` / `DataLoader` (PyTorch).
8. Modelo U-Net 2D.
9. Treinamento.
10. Inferência: gerar um mapa em nova altura.
11. Visualização final.

## 0. Configuração

Todos os parâmetros ficam reunidos aqui. **Edite `DATA_DIR` e a lista de arquivos** para apontar para os seus mapas.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# ---- Caminhos ----------------------------------------------------------
# Pasta onde estão os mapas .txt. Ajuste para o seu ambiente.
DATA_DIR = Path("data")

MAP_FILES = [
    "MAPx260620241658.txt",
    "MAPx270620240930.txt",
    "MAPx020720241338.txt",
    "MAPx030720241005.txt",
    "MAPx040720240950.txt",
]
Z_UM   = [200, 400, 670, 720, 900]          # altura de cada mapa (micrômetros)
Z_LIST = [z * 1e-6 for z in Z_UM]           # em metros

# ---- Parâmetros físicos ------------------------------------------------
DX = DY   = 40e-6     # passo espacial (m) — CONFIRA com o passo real dos seus mapas
ALPHA_DC  = 1e-4      # regularização da downward continuation
SIGMA_PSF = 80e-6     # largura efetiva do footprint do sensor (m)
BETA      = 1e-2      # regularização da deconvolução

# ---- Hiperparâmetros de treino ----------------------------------------
PATCH_SIZE = 48       # tamanho do patch (afeta a resolução das fontes)
STRIDE     = 24       # sobreposição entre patches
BATCH_SIZE = 32
LR         = 1e-4
N_EPOCHS   = 600
MODEL_PATH = "modelo_dc_cnn.pth"

np.random.seed(0)

## 1. Funções físicas (domínio de Fourier)

- **`get_k_grid`** — monta o número de onda radial $k$ (em rad/m).
- **`estimate_z0`** — estimativa heurística da distância sensor–amostra pela
  inclinação do espectro radial (use como referência, não como medida rigorosa).
- **`downward_continuation`** — leva o campo de $z_\text{from}$ a $z_\text{to} < z_\text{from}$
  via $e^{k\,\Delta z}$ com regularização de Tikhonov.
- **`deconvolve_footprint`** — remove um *footprint* gaussiano do sensor.

In [ ]:
def get_k_grid(nx, ny, dx, dy):
    """Número de onda radial k (rad/m) e suas componentes kx, ky."""
    kx = 2 * np.pi * np.fft.fftfreq(nx, d=dx)
    ky = 2 * np.pi * np.fft.fftfreq(ny, d=dy)
    kx, ky = np.meshgrid(kx, ky)
    k = np.sqrt(kx**2 + ky**2)
    return kx, ky, k


def estimate_z0(bz, dx, dy, n_bins=250):
    """Estimativa heurística de z0 pela inclinação do espectro radial (log)."""
    ny, nx = bz.shape
    _, _, k = get_k_grid(nx, ny, dx, dy)
    bz_fft = np.fft.fftshift(np.fft.fftn(bz))

    bins = np.linspace(0, k.max(), n_bins)
    rk, rb = [], []
    for i in range(len(bins) - 1):
        mask = (k >= bins[i]) & (k < bins[i + 1])
        if np.any(mask):
            rk.append((bins[i] + bins[i + 1]) / 2)
            rb.append(np.mean(np.abs(bz_fft[mask])))
    rk, rb = np.array(rk), np.array(rb)

    valid = (rk > 0) & (rb > 0)
    coef = np.polyfit(rk[valid], np.log(rb[valid]), 1)
    return coef[0]


def downward_continuation(bz, z_from, z_to, k, alpha_dc=ALPHA_DC):
    """Continua o campo de z_from para z_to (z_to < z_from)."""
    dz = z_from - z_to
    bz_fft = np.fft.fftshift(np.fft.fftn(bz))
    dc_factor = np.exp(k * dz)
    dc_reg = dc_factor / (1 + alpha_dc * dc_factor**2)
    bz_dc = np.fft.ifftn(np.fft.ifftshift(bz_fft * dc_reg)).real
    return bz_dc


def deconvolve_footprint(bz, k, sigma_psf=SIGMA_PSF, beta=BETA):
    """Deconvolução de um footprint gaussiano do sensor."""
    bz_fft = np.fft.fftshift(np.fft.fftn(bz))
    psf_k = np.exp(-0.5 * (sigma_psf**2) * (k**2))
    bz_deconv = np.fft.ifftn(np.fft.ifftshift(bz_fft / (psf_k + beta))).real
    return bz_deconv

## 2. Carregar e visualizar um mapa

In [ ]:
bz = np.loadtxt(DATA_DIR / MAP_FILES[1])   # exemplo: segundo mapa
Ny, Nx = bz.shape
print("Dimensões do mapa:", Ny, "x", Nx)

plt.figure(figsize=(6, 5))
plt.imshow(bz, cmap="jet", origin="lower")
plt.colorbar(label="Bz (T)")
plt.title("Mapa Bz original")
plt.tight_layout()
plt.show()

z0_est = estimate_z0(bz, DX, DY)
print(f"z0 estimado (heurístico): {z0_est:.3e}")

## 3. Demonstração da *downward continuation*

Leva o mapa para 60 µm mais perto da amostra.

In [ ]:
_, _, k = get_k_grid(Nx, Ny, DX, DY)

z_from = 200e-6
z_to   = z_from - 60e-6        # 60 µm mais perto
bz_dc  = downward_continuation(bz, z_from, z_to, k)

plt.figure(figsize=(6, 5))
plt.imshow(bz_dc, cmap="jet", origin="lower")
plt.colorbar(label="Bz DC (T)")
plt.title(f"Bz após downward continuation ({z_from*1e6:.0f} → {z_to*1e6:.0f} µm)")
plt.tight_layout()
plt.show()

## 4. Conjunto de dados: carregar todos os mapas + *data augmentation*

Os aumentos são fisicamente seguros: *flips*, rotação de 180°, *jitter* de fundo
planar e ruído gaussiano. **Não** se usa rot90/rot270, que alterariam a física.

In [ ]:
raw_maps = [np.loadtxt(DATA_DIR / f) for f in MAP_FILES]
TARGET_SHAPE = raw_maps[0].shape
print("Forma de referência:", TARGET_SHAPE)


def force_shape(bz, target_shape=TARGET_SHAPE):
    """Recorte central para a forma de referência."""
    ny_t, nx_t = target_shape
    ny, nx = bz.shape
    if ny < ny_t or nx < nx_t:
        raise ValueError(f"Imagem menor que a referência: {bz.shape}")
    y0, x0 = (ny - ny_t) // 2, (nx - nx_t) // 2
    return bz[y0:y0 + ny_t, x0:x0 + nx_t]


def augment_geometric(bz):
    return [bz, np.flipud(bz), np.fliplr(bz),
            np.flipud(np.fliplr(bz)), np.rot90(bz, 2)]


def add_background_jitter(bz, amp_ratio=0.03):
    ny, nx = bz.shape
    xx, yy = np.meshgrid(np.linspace(-1, 1, nx), np.linspace(-1, 1, ny))
    amp = amp_ratio * np.max(np.abs(bz))
    a, b, c = np.random.uniform(-amp, amp, size=3)
    return bz + a + b * xx + c * yy


def add_noise(bz, noise_ratio=0.01):
    sigma = noise_ratio * np.std(bz)
    return bz + sigma * np.random.randn(*bz.shape)

In [ ]:
bz_aug, z_aug = [], []

def append_safe(bz, z):
    bz_aug.append(force_shape(bz))
    z_aug.append(z)

for bz_i, z in zip(raw_maps, Z_LIST):
    for bz_g in augment_geometric(bz_i):
        append_safe(bz_g, z)
        if np.random.rand() < 0.5:
            append_safe(add_background_jitter(bz_g), z)
        if np.random.rand() < 0.3:
            append_safe(add_noise(bz_g), z)

bz_maps = np.stack(bz_aug)
z_list  = np.array(z_aug)
Nz, Ny, Nx = bz_maps.shape
print("Dataset aumentado:", Nz, Ny, Nx)

Visualização das variações aplicadas ao primeiro mapa:

In [ ]:
bz0 = raw_maps[0]
maps = [bz0, np.flipud(bz0), np.fliplr(bz0), np.flipud(np.fliplr(bz0)),
        np.rot90(bz0, 2), add_background_jitter(bz0), add_noise(bz0)]
titles = ["Original", "Flip vertical", "Flip horizontal", "Flip duplo",
          "Rotação 180°", "Jitter de fundo", "Ruído"]

plt.figure(figsize=(18, 6))
for i, (m, t) in enumerate(zip(maps, titles)):
    plt.subplot(2, 4, i + 1)
    plt.imshow(m, cmap="jet", origin="lower")
    plt.title(t); plt.axis("off")
    plt.colorbar(fraction=0.046)
plt.suptitle("Variações aplicadas ao primeiro mapa", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Pares entrada → alvo

O **alvo** é o mapa de menor altura (plano de referência). Para cada outro mapa:
downward continuation até o plano de referência → deconvolução → normalização.

In [ ]:
_, _, k = get_k_grid(Nx, Ny, DX, DY)

idx_ref = int(np.argmin(z_list))
z_ref   = z_list[idx_ref]
print(f"Plano de referência: idx={idx_ref}, z_ref={z_ref*1e6:.0f} µm")

bz_ref_target = bz_maps[idx_ref]
scale_t = np.max(np.abs(bz_ref_target)) or 1.0
bz_t_norm = bz_ref_target / scale_t

input_maps, target_maps = [], []
for i in range(Nz):
    if i == idx_ref:
        continue
    bz_dc = downward_continuation(bz_maps[i], z_list[i], z_ref, k)
    bz_dc_deconv = deconvolve_footprint(bz_dc, k)
    scale = np.max(np.abs(bz_dc_deconv)) or 1.0
    input_maps.append(bz_dc_deconv / scale)
    target_maps.append(bz_t_norm)

input_maps  = np.array(input_maps)
target_maps = np.array(target_maps)
print("N_pares =", input_maps.shape[0])

## 6. Patches 2D

In [ ]:
def extract_patches(bz_array, patch_size=PATCH_SIZE, stride=STRIDE):
    patches = []
    n, ny, nx = bz_array.shape
    for idx in range(n):
        img = bz_array[idx]
        for y0 in range(0, ny - patch_size + 1, stride):
            for x0 in range(0, nx - patch_size + 1, stride):
                patches.append(img[y0:y0 + patch_size, x0:x0 + patch_size])
    return np.array(patches)

input_patches  = extract_patches(input_maps)
target_patches = extract_patches(target_maps)
print("input_patches:", input_patches.shape)
print("target_patches:", target_patches.shape)

## 7. `Dataset` e `DataLoader` (PyTorch)

Descomente a linha abaixo se o PyTorch ainda não estiver instalado.

In [ ]:
# !pip install torch torchvision torchaudio

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class BzPatchDataset(Dataset):
    def __init__(self, inputs, targets):
        self.inputs, self.targets = inputs, targets
    def __len__(self):
        return self.inputs.shape[0]
    def __getitem__(self, idx):
        x = torch.from_numpy(self.inputs[idx][None, ...]).float()
        y = torch.from_numpy(self.targets[idx][None, ...]).float()
        return x, y

train_loader = DataLoader(
    BzPatchDataset(input_patches, target_patches),
    batch_size=BATCH_SIZE, shuffle=True,
)

## 8. Modelo U-Net 2D

In [ ]:
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.InstanceNorm2d(out_ch, affine=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.InstanceNorm2d(out_ch, affine=True),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.conv(x)

class UNet2D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1):
        super().__init__()
        self.down1 = DoubleConv(in_ch, 32); self.pool1 = nn.MaxPool2d(2)
        self.down2 = DoubleConv(32, 64);    self.pool2 = nn.MaxPool2d(2)
        self.down3 = DoubleConv(64, 128);   self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(128, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2); self.conv3 = DoubleConv(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2);  self.conv2 = DoubleConv(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2);   self.conv1 = DoubleConv(64, 32)
        self.out_conv = nn.Conv2d(32, out_ch, 1)
    def forward(self, x):
        c1 = self.down1(x)
        c2 = self.down2(self.pool1(c1))
        c3 = self.down3(self.pool2(c2))
        bn = self.bottleneck(self.pool3(c3))
        u3 = self.conv3(torch.cat([self.up3(bn), c3], dim=1))
        u2 = self.conv2(torch.cat([self.up2(u3), c2], dim=1))
        u1 = self.conv1(torch.cat([self.up1(u2), c1], dim=1))
        return self.out_conv(u1)

## 9. Treinamento

Perda L1 (boa para *denoising*).

In [ ]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

model     = UNet2D(in_ch=1, out_ch=1).to(device)
criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=LR)

for epoch in range(N_EPOCHS):
    model.train()
    running = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        running += loss.item() * x.size(0)
    epoch_loss = running / len(train_loader.dataset)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{N_EPOCHS} - Loss: {epoch_loss:.4e}")

Salvar e (re)carregar o modelo:

In [ ]:
torch.save(model.state_dict(), MODEL_PATH)
print("Modelo salvo em:", MODEL_PATH)

model = UNet2D(in_ch=1, out_ch=1).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print("Modelo carregado.")

## 10. Inferência: gerar um mapa em nova altura

A partir de um mapa medido (ex.: 200 µm), gera o mapa em uma altura menor (ex.: 140 µm).
A U-Net exige dimensões múltiplas de 8, daí o recorte.

In [ ]:
def crop_to_mult8(img):
    ny, nx = img.shape
    return img[:(ny // 8) * 8, :(nx // 8) * 8]

# 1) mapa medido + alturas
bz_meas = np.loadtxt(DATA_DIR / MAP_FILES[0])
z_from, z_to = 200e-6, 140e-6
Ny, Nx = bz_meas.shape
_, _, k = get_k_grid(Nx, Ny, DX, DY)

# 2) downward continuation + 3) deconvolução
bz_dc_deconv = deconvolve_footprint(
    downward_continuation(bz_meas, z_from, z_to, k), k
)

# 4) recorte + 5) normalização
bz_crop = crop_to_mult8(bz_dc_deconv)
Ny_c, Nx_c = bz_crop.shape
scale_test = np.max(np.abs(bz_crop)) or 1.0
bz_norm = bz_crop / scale_test

# 6) CNN + 7) desnormalização
with torch.no_grad():
    x_t = torch.from_numpy(bz_norm[None, None, ...]).float().to(device)
    Bz_pred = model(x_t).cpu().numpy()[0, 0] * scale_test

# np.savetxt("Bz_140um_CNN.txt", Bz_pred)   # descomente para salvar

extent = [0, Nx_c * DX * 1000, 0, Ny_c * DY * 1000]   # mm
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, img, title in zip(
    axes, [bz_meas, bz_dc_deconv, Bz_pred],
    [f"Medido em {z_from*1e6:.0f} µm",
     f"DC + Deconv ({z_from*1e6:.0f} → {z_to*1e6:.0f} µm)",
     f"CNN em {z_to*1e6:.0f} µm"],
):
    im = ax.imshow(img, cmap="jet", origin="lower")
    ax.set_title(title); ax.set_xlabel("x (px)"); ax.set_ylabel("y (px)")
    fig.colorbar(im, ax=ax, label="T")
plt.tight_layout()
plt.show()

## 11. Visualização final com realce de contraste

Realce via `tanh`; ajuste o fator e os limites dos eixos conforme a região de interesse.

In [ ]:
contrast = 2          # regula o contraste do tanh
gain     = 12         # ganho de amplitude para visualização

bz_n = bz_dc_deconv / np.max(np.abs(bz_dc_deconv))
bz_show = np.tanh(bz_n * contrast) * gain
vlim = np.max(np.abs(bz_show))

plt.figure(figsize=(10, 7))
plt.imshow(bz_show, cmap="jet", origin="lower",
           extent=extent, vmin=-vlim, vmax=vlim)
plt.colorbar(fraction=0.28, shrink=0.8)
plt.xlabel("X (mm)", fontsize=22)
plt.ylabel("Y (mm)", fontsize=22)
plt.tick_params(axis="both", which="major", labelsize=22)
plt.tight_layout()
plt.show()